# Diabetes Prediction using an Artificial Neural Network (ANN)

**Goal:** Build an end-to-end Deep Learning classification pipeline using TensorFlow/Keras to predict whether a patient is likely to have diabetes, based on the Pima Indians Diabetes Dataset.

This notebook covers:
1. Data Preparation
2. Building the ANN
3. Model Training
4. Model Evaluation


## Part 1 — Data Preparation

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# For reproducibility
np.random.seed(42)

pd.set_option('display.max_columns', None)


### 1.1 Load the Dataset

In [ ]:
# Load the dataset (make sure diabetes.csv is in the same folder as this notebook)
df = pd.read_csv('diabetes.csv')

df.head()


### 1.2 Explore the Dataset

In [ ]:
# a. Shape of the dataset
print("Shape of the dataset (rows, columns):", df.shape)


In [ ]:
# b. Column names
print("Column names:")
print(df.columns.tolist())


In [ ]:
# c. Data types
df.dtypes


In [ ]:
# d. Missing values
print("Missing values per column:")
print(df.isnull().sum())


**Note:** In this dataset, missing values are often encoded as `0` in columns like `Glucose`,
`BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` (a value of 0 is not physiologically possible
for these attributes). Let's check for these hidden missing values and handle them by replacing
zeros with the column median.

In [ ]:
# Check for biologically impossible zero values in key columns
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Count of zero values (treated as missing) in each column:")
print((df[zero_cols] == 0).sum())


In [ ]:
# Replace zeros with NaN, then impute with the median of each column
df_clean = df.copy()
df_clean[zero_cols] = df_clean[zero_cols].replace(0, np.nan)

print("Missing values after replacing zeros with NaN:")
print(df_clean.isnull().sum())

# Impute missing values using the median
for col in zero_cols:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

print("\nMissing values after imputation:")
print(df_clean.isnull().sum())


In [ ]:
# Quick statistical summary
df_clean.describe()


In [ ]:
# Class balance of the target variable
print(df_clean['Outcome'].value_counts())

sns.countplot(x='Outcome', data=df_clean)
plt.title('Class distribution: 0 = No Diabetes, 1 = Diabetes')
plt.show()


### 1.3 Separate Features (X) and Target (y)

In [ ]:
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

print("Features shape:", X.shape)
print("Target shape:", y.shape)


### 1.4 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


### 1.5 Feature Scaling

Neural networks converge faster and more reliably when input features are on a similar scale.
We use `StandardScaler` to standardize features by removing the mean and scaling to unit variance.

Important: the scaler is **fit only on the training data** and then used to transform both the
training and test data, to avoid data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Sample of scaled training data:")
print(X_train_scaled[:5])


**Part 1 Deliverables Summary**
- Dataset shape, column names, data types were explored above.
- Hidden missing values (zeros in medical columns) were identified and imputed with the median.
- Features (`X`) and target (`y`, the `Outcome` column) were separated.
- Data was split into training (80%) and testing (20%) sets using stratified sampling.
- Features were standardized using `StandardScaler`.

## Part 2 — Building the Artificial Neural Network

In [ ]:
# Import TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

print("TensorFlow version:", tf.__version__)

tf.random.set_seed(42)


### ANN Architecture

We build a fully connected feed-forward neural network (Sequential model) with:

| Layer | Neurons | Activation | Purpose |
|---|---|---|---|
| Input Layer | 8 (equal to number of features) | — | Receives the 8 scaled input features |
| Hidden Layer 1 | 16 | ReLU | Learns non-linear combinations of input features |
| Hidden Layer 2 | 8 | ReLU | Learns higher-level non-linear patterns |
| Output Layer | 1 | Sigmoid | Outputs probability of diabetes (0 to 1) |

**Why these choices?**
- **ReLU (Rectified Linear Unit)** is used in hidden layers because it is computationally efficient,
  helps avoid the vanishing gradient problem, and works well for most tabular data problems.
- **Sigmoid** is used in the output layer because this is a **binary classification** problem
  (Outcome = 0 or 1). Sigmoid squashes the output into a probability between 0 and 1.


In [ ]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),        # Input Layer: 8 features
    Dense(16, activation='relu', name='hidden_layer_1'),  # First Hidden Layer
    Dense(8, activation='relu', name='hidden_layer_2'),   # Second Hidden Layer
    Dense(1, activation='sigmoid', name='output_layer')   # Output Layer
])

model.summary()


### Compiling the Model

- **Optimizer: Adam** — an adaptive learning-rate optimizer that combines the advantages of
  Momentum and RMSProp; it converges quickly and works well without extensive tuning.
- **Loss Function: Binary Crossentropy** — the standard loss function for binary classification
  problems, measuring the difference between predicted probabilities and actual binary labels.
- **Metric: Accuracy** — used to monitor the proportion of correctly classified samples during
  training and evaluation.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


## Part 3 — Model Training

We train the model using the scaled training data. A portion of the training data (20%) is
held out as a validation set during training so we can monitor for overfitting.

- **Epochs:** 100 — the model sees the entire training dataset 100 times. This is generally enough
  for a small tabular dataset like this to converge without excessive overfitting.
- **Batch size:** 16 — number of samples processed before the model's weights are updated.

In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    verbose=1
)


In [ ]:
# Final training accuracy
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print(f"Final Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")


In [ ]:
# Plot training & validation accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


**Part 3 Deliverables Summary**
- The model was trained for **100 epochs** with a batch size of 16.
- Training and validation accuracy/loss curves are plotted above to visualize convergence and
  check for overfitting.

## Part 4 — Model Evaluation

### 4.1 Generate Predictions on the Test Set

In [ ]:
# Predict probabilities on test data
y_pred_prob = model.predict(X_test_scaled)

# Convert probabilities into binary class labels using a 0.5 threshold
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Sample predicted probabilities:", y_pred_prob[:10].flatten())
print("Sample predicted classes:      ", y_pred[:10])
print("Sample actual classes:         ", y_test.values[:10])


### 4.2 Accuracy Score

In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Also using Keras' built-in evaluate
loss, keras_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Keras evaluate() -> Loss: {loss:.4f}, Accuracy: {keras_acc:.4f}")


### 4.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes (0)', 'Diabetes (1)'],
            yticklabels=['No Diabetes (0)', 'Diabetes (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Detailed classification report (Precision, Recall, F1-score)
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))


### 4.4 Interpretation of Results

- **True Positives (bottom-right cell):** Patients correctly predicted to have diabetes.
- **True Negatives (top-left cell):** Patients correctly predicted to not have diabetes.
- **False Positives (top-right cell):** Patients incorrectly predicted to have diabetes (Type I error).
- **False Negatives (bottom-left cell):** Patients incorrectly predicted as not having diabetes,
  even though they do (Type II error) — this is the more clinically costly error, since a missed
  diabetes diagnosis can delay treatment.

Overall, the ANN achieves a reasonable test accuracy on this dataset. Performance could likely be
improved further by:
- Tuning the number of hidden layers/neurons
- Trying different epoch counts, batch sizes, or learning rates
- Using dropout layers or regularization to reduce overfitting
- Addressing class imbalance (e.g., using class weights or resampling techniques)
- Performing more thorough feature engineering
